In [41]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Étape 1: Charger les données
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Étape 2: Vérification des colonnes disponibles
print(train_df.columns)  # Vérifier les noms de colonnes dans train.csv
print(test_df.columns)   # Vérifier les noms de colonnes dans test.csv

# Étape 3: Prétraitement des données (train.csv)
# Remplacer les NaN par 0 dans les colonnes de MisconceptionId
train_df['MisconceptionAId'] = train_df['MisconceptionAId'].fillna(0)
train_df['MisconceptionBId'] = train_df['MisconceptionBId'].fillna(0)
train_df['MisconceptionCId'] = train_df['MisconceptionCId'].fillna(0)
train_df['MisconceptionDId'] = train_df['MisconceptionDId'].fillna(0)

# Initialisation du vectoriseur TF-IDF
vectorizer = TfidfVectorizer()

# Combiner les colonnes textuelles en une seule avant d'ajuster le vectoriseur
combined_text = pd.concat([train_df['AnswerAText'], train_df['AnswerBText'],
                           train_df['AnswerCText'], train_df['AnswerDText']], axis=0)

# Ajuster le vectoriseur sur toutes les réponses textuelles combinées
vectorizer.fit(combined_text)

# Appliquer le vectoriseur sur les réponses du train set
X_train_text_A = vectorizer.transform(train_df['AnswerAText'])
X_train_text_B = vectorizer.transform(train_df['AnswerBText'])
X_train_text_C = vectorizer.transform(train_df['AnswerCText'])
X_train_text_D = vectorizer.transform(train_df['AnswerDText'])

# Convertir les matrices sparse en DataFrame
X_train_text_A_df = pd.DataFrame(X_train_text_A.toarray(), columns=vectorizer.get_feature_names_out())
X_train_text_B_df = pd.DataFrame(X_train_text_B.toarray(), columns=vectorizer.get_feature_names_out())
X_train_text_C_df = pd.DataFrame(X_train_text_C.toarray(), columns=vectorizer.get_feature_names_out())
X_train_text_D_df = pd.DataFrame(X_train_text_D.toarray(), columns=vectorizer.get_feature_names_out())

# Préparation des données d'entraînement
X = train_df[['ConstructId', 'SubjectId']]  # Utilisation des colonnes pertinentes
X = pd.concat([X, X_train_text_A_df], axis=1)  # Ajouter les caractéristiques textuelles pour A
X = pd.concat([X, X_train_text_B_df], axis=1)  # Ajouter les caractéristiques textuelles pour B
X = pd.concat([X, X_train_text_C_df], axis=1)  # Ajouter les caractéristiques textuelles pour C
X = pd.concat([X, X_train_text_D_df], axis=1)  # Ajouter les caractéristiques textuelles pour D

y_A = train_df['MisconceptionAId']  # Label pour la prédiction de MisconceptionAId
y_B = train_df['MisconceptionBId']  # Label pour la prédiction de MisconceptionBId
y_C = train_df['MisconceptionCId']  # Label pour la prédiction de MisconceptionCId
y_D = train_df['MisconceptionDId']  # Label pour la prédiction de MisconceptionDId

# Diviser en jeu d'entraînement et validation
X_train, X_val, y_train_A, y_val_A = train_test_split(X, y_A, test_size=0.2, random_state=42)
X_train, X_val, y_train_B, y_val_B = train_test_split(X, y_B, test_size=0.2, random_state=42)
X_train, X_val, y_train_C, y_val_C = train_test_split(X, y_C, test_size=0.2, random_state=42)
X_train, X_val, y_train_D, y_val_D = train_test_split(X, y_D, test_size=0.2, random_state=42)

# Créer et entraîner les modèles de RandomForest pour chaque MisconceptionId
model_A = RandomForestClassifier(n_estimators=100, random_state=42)
model_A.fit(X_train, y_train_A)

model_B = RandomForestClassifier(n_estimators=100, random_state=42)
model_B.fit(X_train, y_train_B)

model_C = RandomForestClassifier(n_estimators=100, random_state=42)
model_C.fit(X_train, y_train_C)

model_D = RandomForestClassifier(n_estimators=100, random_state=42)
model_D.fit(X_train, y_train_D)

# Prétraitement des réponses dans le jeu de test (test.csv)
X_test_text_A = vectorizer.transform(test_df['AnswerAText'])
X_test_text_B = vectorizer.transform(test_df['AnswerBText'])
X_test_text_C = vectorizer.transform(test_df['AnswerCText'])
X_test_text_D = vectorizer.transform(test_df['AnswerDText'])

# Convertir en DataFrame pour faciliter l'intégration
X_test_text_A_df = pd.DataFrame(X_test_text_A.toarray(), columns=vectorizer.get_feature_names_out())
X_test_text_B_df = pd.DataFrame(X_test_text_B.toarray(), columns=vectorizer.get_feature_names_out())
X_test_text_C_df = pd.DataFrame(X_test_text_C.toarray(), columns=vectorizer.get_feature_names_out())
X_test_text_D_df = pd.DataFrame(X_test_text_D.toarray(), columns=vectorizer.get_feature_names_out())

# Ajouter les caractéristiques textuelles aux autres caractéristiques
X_test_input = test_df[['ConstructId', 'SubjectId']].copy()
X_test_input = pd.concat([X_test_input, X_test_text_A_df], axis=1)
X_test_input = pd.concat([X_test_input, X_test_text_B_df], axis=1)
X_test_input = pd.concat([X_test_input, X_test_text_C_df], axis=1)
X_test_input = pd.concat([X_test_input, X_test_text_D_df], axis=1)

# Prédictions sur le jeu de test
test_df['MisconceptionIds_A'] = model_A.predict(X_test_input)
test_df['MisconceptionIds_B'] = model_B.predict(X_test_input)
test_df['MisconceptionIds_C'] = model_C.predict(X_test_input)
test_df['MisconceptionIds_D'] = model_D.predict(X_test_input)

# Affichage des premières lignes du jeu de test avec les prédictions
print(test_df[['QuestionId', 'MisconceptionIds_A', 'MisconceptionIds_B', 'MisconceptionIds_C', 'MisconceptionIds_D']].head())



Index(['QuestionId', 'ConstructId', 'ConstructName', 'SubjectId',
       'SubjectName', 'CorrectAnswer', 'QuestionText', 'AnswerAText',
       'AnswerBText', 'AnswerCText', 'AnswerDText', 'MisconceptionAId',
       'MisconceptionBId', 'MisconceptionCId', 'MisconceptionDId'],
      dtype='object')
Index(['QuestionId', 'ConstructId', 'ConstructName', 'SubjectId',
       'SubjectName', 'CorrectAnswer', 'QuestionText', 'AnswerAText',
       'AnswerBText', 'AnswerCText', 'AnswerDText'],
      dtype='object')
   QuestionId  MisconceptionIds_A  MisconceptionIds_B  MisconceptionIds_C  \
0        1869                 0.0                 0.0                 0.0   
1        1870                 0.0                 0.0              1755.0   
2        1871              1287.0                 0.0              1287.0   

   MisconceptionIds_D  
0              1672.0  
1               167.0  
2              1073.0  


In [43]:
# Exemple d'itération avec l'ajout de la réponse dans les caractéristiques
submission = []

# Parcours du DataFrame de test
for index, row in test_df.iterrows():
    question_id = row['QuestionId']
    
    # Traiter chaque réponse possible (A, B, C, D)
    for answer_column, answer_text_column in zip(
        ['AnswerA', 'AnswerB', 'AnswerC', 'AnswerD'],  # Colonnes de réponse (A, B, C, D)
        ['AnswerAText', 'AnswerBText', 'AnswerCText', 'AnswerDText']  # Colonnes du texte de réponse
    ):
        answer = row[answer_text_column]
        
        # Construire les caractéristiques pour chaque réponse
        X_input = row[['ConstructId', 'SubjectId']].copy()
        # Ajouter le texte de la réponse comme caractéristique
        # Ajouter le texte de la réponse comme caractéristique
        X_input['AnswerText'] = answer  # Ajouter le texte de la réponse comme caractéristique

        # Appliquer la vectorisation sur la nouvelle colonne 'AnswerText'
        X_input_transformed = vectorizer.transform([X_input['AnswerText']])  # Transformer le texte dans une liste

        # Vérifier si X_input est un DataFrame
        if isinstance(X_input, pd.Series):
            X_input = X_input.to_frame().T  # Convertir en DataFrame si nécessaire (transpose pour avoir une ligne)

        # Convertir X_input en un DataFrame, y ajouter les caractéristiques d'origine
        X_input_final = pd.concat([X_input.drop('AnswerText', axis=1), pd.DataFrame(X_input_transformed.toarray(), columns=vectorizer.get_feature_names_out())], axis=1)

        # S'assurer que les caractéristiques sont dans le même ordre que lors de l'entraînement
        # Récupérer l'ordre des colonnes utilisé lors de l'entraînement du modèle (supposons que vous avez sauvegardé les colonnes d'entraînement)
        # Par exemple, 'feature_columns' serait la liste des colonnes utilisées lors de l'entraînement
        feature_columns = vectorizer.get_feature_names_out()  # ou toute autre liste de colonnes utilisées pendant l'entraînement

        # Réorganiser les colonnes de X_input_final pour qu'elles correspondent à l'ordre d'entraînement
        X_input_final = X_input_final[feature_columns]

        # Ajouter les colonnes manquantes avec des valeurs par défaut ou des valeurs récupérées
        # Si 'ConstructId' et 'SubjectId' sont des colonnes importantes, vous devez les ajouter à X_input_final
        if 'ConstructId' not in X_input_final.columns:
            X_input_final['ConstructId'] = 0  # Remplacer 0 par une valeur par défaut ou appropriée

        if 'SubjectId' not in X_input_final.columns:
            X_input_final['SubjectId'] = 0  # Remplacer 0 par une valeur par défaut ou appropriée

        # Réorganiser les colonnes pour qu'elles correspondent à l'ordre attendu par le modèle
        X_input_final = X_input_final[feature_columns]

        # Prédire les misconceptions pour chaque réponse avec les caractéristiques modifiées
        misconceptions_A = model_A.predict(X_input_final)  # Modèle pour MisconceptionAId
        misconceptions_B = model_B.predict(X_input_final)  # Modèle pour MisconceptionBId
        misconceptions_C = model_C.predict(X_input_final)  # Modèle pour MisconceptionCId
        misconceptions_D = model_D.predict(X_input_final)  # Modèle pour MisconceptionDId




        # Combiner les prédictions des MisconceptionIds en une liste
        misconception_ids = []
        if misconceptions_A is not None:
            misconception_ids.extend(map(int, misconceptions_A))  # Convertir en entiers
        if misconceptions_B is not None:
            misconception_ids.extend(map(int, misconceptions_B))  # Convertir en entiers
        if misconceptions_C is not None:
            misconception_ids.extend(map(int, misconceptions_C))  # Convertir en entiers
        if misconceptions_D is not None:
            misconception_ids.extend(map(int, misconceptions_D))  # Convertir en entiers

        # Limiter à 25 MisconceptionIds max, éviter les doublons
        misconception_ids = list(set(misconception_ids))[:25]  # Supprimer les doublons et limiter à 25

        # Créer la ligne de soumission au format "QuestionId_Answer,MisconceptionIds"
        submission.append(f"{question_id}_{answer_column[7]},{ ' '.join(map(str, misconception_ids)) }")  # '_A', '_B', etc.

# Sauvegarder les résultats dans un fichier CSV
with open('submission.csv', 'w') as f:
    f.write("QuestionId_Answer,MisconceptionId\n")  # En-tête
    f.write("\n".join(submission))  # Écrire toutes les lignes de la soumission




ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- ConstructId
- SubjectId


In [51]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Chargement des données
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Prétraitement des données (train.csv)
train_df['MisconceptionAId'] = train_df['MisconceptionAId'].fillna(0)
train_df['MisconceptionBId'] = train_df['MisconceptionBId'].fillna(0)
train_df['MisconceptionCId'] = train_df['MisconceptionCId'].fillna(0)
train_df['MisconceptionDId'] = train_df['MisconceptionDId'].fillna(0)

# Initialisation du vectoriseur TF-IDF
vectorizer = TfidfVectorizer()

# Combiner les colonnes textuelles en une seule avant d'ajuster le vectoriseur
combined_text = pd.concat([train_df['AnswerAText'], train_df['AnswerBText'],
                           train_df['AnswerCText'], train_df['AnswerDText']], axis=0)

# Ajuster le vectoriseur sur toutes les réponses textuelles combinées
vectorizer.fit(combined_text)

# Appliquer le vectoriseur sur les réponses du train set
X_train_text_A = vectorizer.transform(train_df['AnswerAText'])
X_train_text_B = vectorizer.transform(train_df['AnswerBText'])
X_train_text_C = vectorizer.transform(train_df['AnswerCText'])
X_train_text_D = vectorizer.transform(train_df['AnswerDText'])

# Convertir les matrices sparse en DataFrame
X_train_text_A_df = pd.DataFrame(X_train_text_A.toarray(), columns=vectorizer.get_feature_names_out())
X_train_text_B_df = pd.DataFrame(X_train_text_B.toarray(), columns=vectorizer.get_feature_names_out())
X_train_text_C_df = pd.DataFrame(X_train_text_C.toarray(), columns=vectorizer.get_feature_names_out())
X_train_text_D_df = pd.DataFrame(X_train_text_D.toarray(), columns=vectorizer.get_feature_names_out())

# Concatenation des réponses textuelles en une seule matrice
X_train_text_combined = pd.concat([X_train_text_A_df, X_train_text_B_df, 
                                   X_train_text_C_df, X_train_text_D_df], axis=1)

# Ajout des colonnes 'ConstructId' et 'SubjectId'
X_train = pd.concat([train_df[['ConstructId', 'SubjectId']], X_train_text_combined], axis=1)

# Labels
y_train = train_df[['MisconceptionAId', 'MisconceptionBId', 'MisconceptionCId', 'MisconceptionDId']]

# Diviser en jeu d'entraînement et validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Créer et entraîner le modèle RandomForest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Prétraitement des réponses dans le jeu de test (test.csv)
X_test_text_A = vectorizer.transform(test_df['AnswerAText'])
X_test_text_B = vectorizer.transform(test_df['AnswerBText'])
X_test_text_C = vectorizer.transform(test_df['AnswerCText'])
X_test_text_D = vectorizer.transform(test_df['AnswerDText'])

# Convertir en DataFrame pour faciliter l'intégration
X_test_text_A_df = pd.DataFrame(X_test_text_A.toarray(), columns=vectorizer.get_feature_names_out())
X_test_text_B_df = pd.DataFrame(X_test_text_B.toarray(), columns=vectorizer.get_feature_names_out())
X_test_text_C_df = pd.DataFrame(X_test_text_C.toarray(), columns=vectorizer.get_feature_names_out())
X_test_text_D_df = pd.DataFrame(X_test_text_D.toarray(), columns=vectorizer.get_feature_names_out())

# Concatenation des réponses textuelles dans le jeu de test
X_test_text_combined = pd.concat([X_test_text_A_df, X_test_text_B_df, 
                                  X_test_text_C_df, X_test_text_D_df], axis=1)

# Ajout des colonnes 'ConstructId' et 'SubjectId' pour le test
X_test_input = pd.concat([test_df[['ConstructId', 'SubjectId']], X_test_text_combined], axis=1)

# Prédictions sur le jeu de test
predictions = model.predict(X_test_input)

# Ajout des prédictions dans le dataframe de test
test_df['MisconceptionIds_A'] = predictions[:, 0]
test_df['MisconceptionIds_B'] = predictions[:, 1]
test_df['MisconceptionIds_C'] = predictions[:, 2]
test_df['MisconceptionIds_D'] = predictions[:, 3]

# Affichage des premières lignes du jeu de test avec les prédictions
print(test_df[['QuestionId', 'MisconceptionIds_A', 'MisconceptionIds_B', 'MisconceptionIds_C', 'MisconceptionIds_D']].head())


   QuestionId  MisconceptionIds_A  MisconceptionIds_B  MisconceptionIds_C  \
0        1869                 0.0                 0.0                 0.0   
1        1870                 0.0                 0.0              1755.0   
2        1871              1287.0                 0.0              1287.0   

   MisconceptionIds_D  
0              1672.0  
1               167.0  
2              1073.0  


In [58]:
# Fonction pour s'assurer qu'il y a exactement 25 valeurs, avec des valeurs par défaut si nécessaire
def ensure_25_predictions(predictions):
    if isinstance(predictions, float):  # Si c'est une valeur unique ou NaN
        predictions = [int(predictions)]
    elif isinstance(predictions, str):  # Si les prédictions sont une chaîne
        predictions = predictions.split()
    if len(predictions) < 25:  # Compléter si nécessaire
        predictions.extend([0] * (25 - len(predictions)))
    return predictions[:25]

# Créer un DataFrame pour le fichier de soumission
submission = []

# Ajouter les lignes pour chaque question et chaque réponse incorrecte
for idx, row in test_df.iterrows():
    correct_answer = row['CorrectAnswer']  # Identifier la bonne réponse (par exemple, 'A', 'B', 'C', ou 'D')

    # Parcourir toutes les réponses possibles
    for answer in ['A', 'B', 'C', 'D']:
        if answer == correct_answer:  # Ignorer la bonne réponse
            continue

        question_answer_id = f"{row['QuestionId']}_{answer}"

        # Récupérer les prédictions pour ce MisconceptionId
        misconception_id_col = f'MisconceptionIds_{answer}'  # Supposons que les prédictions sont stockées avec ce schéma
        misconception_ids = row.get(misconception_id_col, [0])  # Valeur par défaut si la colonne manque

        # S'assurer qu'il y a exactement 25 prédictions
        predictions = ensure_25_predictions(misconception_ids)

        # Formatage de la sortie pour ce QuestionId_Answer
        formatted_ids = ' '.join(map(str, predictions))

        # Ajouter à la liste de soumission
        submission.append([question_answer_id, formatted_ids])

# Créer le DataFrame de soumission
submission_df = pd.DataFrame(submission, columns=['QuestionId_Answer', 'MisconceptionId'])

# Sauvegarder le fichier CSV
submission_df.to_csv('submission.csv', index=False)
